In [81]:
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.optimize import fsolve, differential_evolution,minimize

In [82]:
def m_func(value,Delta, Log_space=False):
    if Log_space:
        return np.ceil(np.exp(value*Delta))
    else:
        return np.ceil(np.power(value,Delta))


In [83]:
def s_func(value,Epsilon,Log_space=False):
    if Log_space:
        return int(np.ceil(Epsilon*np.exp(value)))
    else:
        return int(np.ceil(Epsilon*value))

Requirement 1, lower bound on $\delta$, depending on $\epsilon$

In [84]:
def delta_bound(Epsilon):
    return 1/(1-Epsilon)*3/4

def epsilon_bound(Delta):
    return 1 - 3/(4*Delta)


Requirement 2 on the value of $\log_2n$, the largest $\delta$ is, the smaller n becomes,
                                 the further $\epsilon$ is from its optimum, the largest n is

Optimum value of epsilon is $1/2 - \frac{3}{8\delta}$

In [85]:
def optimal_epsilon(Delta):
    return 1/2 - 3/(8*Delta)

def req_on_log2n(Delta,Epsilon):
    return 1/(2*Delta*Epsilon*(1-Epsilon) - 3*Epsilon/2)


Bound on $\mathbb{P}[D_R$ does not hold $]$

In [86]:
def bound_on_Dx(m1,Omega):
    return m1*np.exp(-np.sqrt(m1)/(3*np.log(m1)**Omega))

# for m in np.logspace(1,15,1000):
#     if bound_on_Dx(m) < 1e-18:
#         print(f"m = {m:.4e}")
#         for n in np.logspace(1,100,1000):
#             if m_func(n,delta) >= m:
#                 print(f"n = {n:.4e}")
#                 break
#         break

Bound on $\mathbb{P}[F_R$ does not hold $]$

In [87]:
def bound_on_Fx(value,Delta,Log_space=False):
    if Log_space:
        M1 = m_func(value,Delta, Log_space)
        return M1*np.exp(-np.exp(value*(1-Delta))/4)
    else:
        return m_func(value,Delta)*np.exp(-np.power(value,1-Delta)/4)

# for n in np.logspace(1,100,1000):
#     result = bound_on_Fx(n,delta)
#     if result < 1e-3:
#         print(f"Value of n = {n:.1e}, result is {result:2e}")
#         break

Bound on $P[T_R$ does not hold$]$, given by $\frac{1}{m}$

In [88]:
def bound_on_Tx(m1,K):
    return np.power(m1,-K/6+2)

# for n in np.logspace(1,300,1000):
#
#     if 1/m_func(n,delta)< 1e-12:
#         print(f"Value of n = {n:.1e}")
#         break

Bound on $\mathbb{P}[\mathcal{E}(\epsilon)^c]$

In [89]:
def bound_on_Eepsilon(value,Delta,Epsilon,Log_space=True,c0= 1+1e-30):

    beta1 = 2*c0*Epsilon*np.log(3) + 5*c0*Epsilon*np.log(Epsilon) + 2*np.log(2) + c0*Epsilon*np.log(c0)
    beta2 = 3*c0*Epsilon
    beta3 = 4*(Epsilon-1)*Epsilon*c0
    m = m_func(value,Delta,Log_space)
    if Log_space:
        return np.exp(value)*beta1 + np.exp(value)*value*beta2 + np.exp(value)*np.log(m)*beta3
    # else:
    #     s = s_func(value,Epsilon,Log_space=False)
    #     m1 = m_func(value,Delta,Log_space=False)
    #     return np.power(3*Epsilon**2*value,2*s)*np.power(m1,(Epsilon-1)*4*s)*np.power(2,2*value)*np.power(s,s)

From here, calculations for Lemma 4.4, first one is bound on $\mathbb{P}[\mathbb{H}$ doesnt hold$]$

In [90]:
def bound_on_H(value,Delta,Gamma,Constant1,Log_space=False):
    term = Gamma*Constant1
    if Log_space: #if in log space, returns only the exponent
        M1 = m_func(value,Delta,Log_space)
        return -term*np.exp(2*value)/M1
    else:
        m1 = m_func(value,Delta,Log_space)
        return np.exp(-term*np.power(value,2)/m1)


Bound on P[there is no $\{e,f\} \in M$ st no triangle in $H_R$ contains $e$ or $f | H_R = H_R^*] := A$

The bound on A depends on a parameter $\zeta$, where it satisfies $(1-\frac{1}{\log^\omega(m)})^{4k\log(m)} \geq \zeta$

I do computations in log space, to get an estimate of n, ($L = \log n$)

H_condition checks that $2\log(2) + \epsilon\log(\epsilon) + \epsilon - \alpha_1 e^L / M < 0$. The term on the left is multiplied by $e^L$ and then we take the exponential, thus when the latter condition is satisfied, the bound on $P[H^c]$ becomes really small

A_condition checks that $2\log(2) + \epsilon\log(\epsilon) + \epsilon L - \frac{\gamma}{C_k} \frac{e^L}{M\log^{2+2\omega}(M)} \bigg( 1-\frac{1}{log^\omega(M)} \bigg)^{4k\log(M)}<0 $, and the argument is the same as above

In [91]:
def H_condition(value,Delta,Gamma,Epsilon,Constant1=3/416,c0 = 1+1e-30):#constant 1 is the value that arises from lemma 7.2
    Alpha1 = Gamma*Constant1
    M1 = m_func(value,Delta,Log_space=True)
    Term1 = 2*np.log(2)*c0 +c0*Epsilon*np.log(Epsilon) + c0*Epsilon*value + c0*Epsilon*np.log(c0)
    Term2 = Alpha1*np.exp(value)/M1
    return Term1-Term2

def A_condition(value,Delta,Epsilon,Gamma,Omega,K):
    C_k = 16*(4*K+1)*(2*K+1) + 16
    c0 = 1+1e-30
    M1 = m_func(value,Delta,Log_space=True)


    Term1 = 2*np.log(2)*c0 +c0*Epsilon*np.log(Epsilon) + c0*Epsilon*value + c0*Epsilon*np.log(c0)

    Term2 = Gamma/C_k * np.exp(value) /(M1*np.power(np.log(M1),2+2*Omega)) * np.power(1-1/np.log(M1)**Omega,4*k_param*np.log(M1))

    return Term1-Term2




Computations for theorem 3.1

Requirements on n, $(1-\frac{1}{\log^2(m)})^{80\log(m)} \geq \zeta$, from lemma 4.4 and thm3.1, and the requirement from lemma4.2, namely $2+m \leq \epsilon^2n$

In [92]:
def thm31_req(value,Delta,Epsilon, Omega, bound,Log_space=True):
    if Log_space:
        M1 = m_func(value,Delta,Log_space=Log_space)
        return bound > 4/M1 + 16/np.power(np.log(M1),2*Omega) + Epsilon**2/(4*np.exp(value))

def lemma44_req(value,K,Omega,Zeta):
    return np.power(1-1/value**Omega,4*K*value) >= Zeta

def lemma42_req(value,Delta,Epsilon,Log_space=True):
    M1 = m_func(value, Delta, Log_space)
    if Log_space:
        return 2 + M1    <= Epsilon**2*np.exp(value)

In [93]:
def least_L_with_params(delta, epsilon, gamma, omega, k, tol=1e-9):
    # Assuming epsilon and delta_bound are defined
    params = {
        "delta": delta,
        "gamma": gamma,
        "k": k
    }

    # Define requirements as (Current Value, Boolean Condition, Description)
    requirements = {
        "Delta": (delta, 1 > delta > max(delta_bound(epsilon), 0.8), f"max({delta_bound(epsilon):.2f}, 0.8) < δ < 1"),
        "Gamma": (f"{gamma:.2e}", 0 < gamma < epsilon ** 4 / 4, f"0 < γ < {epsilon**4 / 4:.2e}"),
        "k": (k, k >12 , "k > 12")
    }

    # Check for failures
    failed = [name for name, (val, met, desc) in requirements.items() if not met]

    if failed:
        # print("═══ PARAMETER CHECK FAILED ═══")
        # for name, (val, met, desc) in requirements.items():
        #     status = "✓" if met else "✗"
        #     print(f"{status} {name:6} : {val:<8} | Required: {desc}")
        #
        # print(f"\nNote: epsilon can be at most: {1 - 3/(4*delta):.4f}")
        return 1e12
        # raise ValueError(f"Constraint violation in: {', '.join(failed)}")

    # print("✓ All parameters satisfy requirements.")

    # Define a helper to check all conditions for a given L
    def is_feasible(L):
        H_exp = H_condition(value=L, Delta=delta, Gamma=gamma, Constant1=3/416, Epsilon=epsilon)
        A_exp = A_condition(value=L, Delta=delta, Epsilon=epsilon, Gamma=gamma, K=k, Omega=omega)

        if not H_exp < 0 or not A_exp < 0:
            return False

        # Calculate bounds
        Fx_bound = bound_on_Fx(value=L, Delta=delta, Log_space=True)
        M = m_func(value=L, Delta=delta, Log_space=True)
        Dx_bound = bound_on_Dx(M, omega)
        Tx_bound = bound_on_Tx(M, k)

        Ee_bound = bound_on_Eepsilon(value=L, Delta=delta, Epsilon=epsilon, Log_space=True)

        conds = [
            thm31_req(L, Delta=delta, Epsilon=epsilon, Omega=omega, bound=epsilon**4/4-gamma),
            L/np.log(2) > req_on_log2n(delta, epsilon),
            lemma42_req(L,Delta=delta,Epsilon=epsilon,Log_space=True),
        ]

        return np.exp(np.exp(L) *A_exp) + np.exp(np.exp(L)*H_exp) + 2*(Fx_bound+Dx_bound+Tx_bound) + np.exp(Ee_bound) <1 and all(conds)
    # Bisection Search
    low = 300
    high = 340

    # First, check if high is even feasible
    if not is_feasible(high):
        return 1e12

    best_L = high
    while (high - low) > tol:
        mid = (low + high) / 2
        if is_feasible(mid):
            best_L = mid
            high = mid  # Try to find a smaller L in the lower half
        else:
            low = mid   # Need a larger L, look in the upper half

    # print(f"logn is around {best_L:.10f} and so n is around {np.exp(best_L):.8e}")
    return best_L

In [94]:
def objective(params):
    # Unpack parameters
    delta, epsilon, gamma, omega, k = params

    # Call your existing bisection-based function
    L = least_L_with_params(delta, epsilon, gamma, omega, k)

    # If the parameters are invalid, least_L_with_params returns 1e12.
    # This acts as a "penalty" that pushes the optimizer away from bad zones.
    return L

In [95]:
k_param=12.1
DELTA = 0.81
EPSILON= optimal_epsilon(DELTA)+0.0011
GAMMA = EPSILON**4/10.4
OMEGA = 1.6
# Bounds for (delta, epsilon, gamma, omega, k)
# Using small buffers (1e-6) to avoid exact boundary issues
bounds = [
    (0.8001, 0.9999),          # delta: 0.8 < delta < 1
    (0.0001, 0.25),              # epsilon: typically small positive
    (1e-12, 0.1),               # gamma: 0 < gamma < epsilon**4 / 4
    (1.0001, 10.0),             # omega: typically > 1
    (12.0001, 50.0)             # k: k > 12
]

# Initial guess (starting point for the optimizer)
x0 = [DELTA, EPSILON, GAMMA, OMEGA, k_param]

# We use COBYLA or Nelder-Mead because your function is likely not differentiable
res = minimize(
    objective,
    x0,
    method='Nelder-Mead',
    bounds=bounds,
    options={'maxiter': 500, 'disp': True}
)

if res.success:
    optimized_delta, optimized_epsilon, optimized_gamma, optimized_omega, optimized_k = res.x
    print(f"Minimum L found: {res.fun:4f}")
    print(f"Value of n : {np.exp(res.fun):.6e}")
    print(f"Optimal delta: {optimized_delta:.6e}")
    print(f"Optimal epsilon: {optimized_epsilon:.6e}")
    print(f"Optimal gamma: {optimized_gamma:.6e}")
    print(f"Optimal omega: {optimized_omega:.6e}")
    print(f"Optimal k: {optimized_k:.6e}")
else:
    print("Optimization failed to converge.")


Optimization terminated successfully.
         Current function value: 307.334963
         Iterations: 189
         Function evaluations: 333
Minimum L found: 307.334963
Value of n : 2.977683e+133
Optimal delta: 8.104629e-01
Optimal epsilon: 3.735373e-02
Optimal gamma: 2.235472e-07
Optimal omega: 1.624117e+00
Optimal k: 1.202317e+01
